
# 노출시간 정하기: fiber 와 fiducial 을 모두 켠 한 장

fiber 와 fiducial 을 같이 켜고 찍은 FITS 한 장에서 peak 을 찾아 **fiducial / fiber / sky·고정 positioner / 기타** 로 나눠 밝기를 본다
(`lab.spot_brightness`). 종류를 나누려면 목표 파일이 필요하다 (살아 있는 fiber 목표 + fiducial 전체 패턴으로 매칭).

- 빛이 안 나는 positioner(`BROKEN_FIBERS`: 분광기 미연결 33, 끊어진 G4 포함) 는 기대하지 않는다. 고정된 positioner(`FIXED_FIBERS`) 와 sky fiber(`SKY_FIBERS`) 는 그 홀의 arm 범위 안에서 목표에 안 붙은 검출을 그것으로 본다.
- 기대했는데 못 찾은 fiber/fiducial 은 `missing` 으로, 어디에도 안 붙은 검출은 `other` 로 (가장 가까운 fiber 목표와 거리 표시; 목표에서 `fib_tol`=3 mm 넘게 벗어난 fiber 가 여기에 온다).
- 포화된 spot 은 실제 peak 을 모르므로 "노출을 x 배 이상 줄여야" 로만 나온다. fiber 와 fiducial 의 밝기 차가 커서 한 노출에 둘 다 포화 없이 담기지 않으면 노출을 나눈다 (`fid_exptime`).

흐름: 3절에 이미지 경로를 넣고 실행 → 종류별 요약·막대 그림 보고 노출 조정 → 다시 찍고 다시 실행 (`history` 에 쌓여 4절에서 비교).


## 촬영 (take_exposure.py)
노출은 `kspec_metrology/notebooks/take_exposure.py` 로 직접 찍는다 (노출시간 [ms]; 이름 끝에 `_0` 이 자동으로 붙는다; `--dry-run` 은 이름만 확인).
```bash
python kspec_metrology/notebooks/take_exposure.py --exptime 400 --out /data/fib_400ms.fits    # fiber + fiducial 을 모두 켜고
```
→ `fib_400ms_0.fits`. 노출을 바꿔 다시 찍고 3절을 다시 실행한다.


In [ ]:

import os, sys, json, glob
PKG_DIR  = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))   # 이 노트북이 kspec_metrology/notebooks/ 에 있을 때 저장소 루트
OUT_DIR  = os.path.expanduser('~/metrology_out')                   # <- 결과를 쓸 폴더

# ---- 설정: 바꾸면 커널을 재시작한다 (표와 제외 목록은 import 때 읽힌다) ----
FIBER_TABLE   = None            # None = 패키지 기본 표 (config73). 다른 표면 경로
# 빈 리스트 [] 는 패키지 기본값(analysis/fiber_config.py, 설치표 2026-10-12 기준) 을 그대로 쓴다. 적으면 그것이 우선.
BROKEN_FIBERS = []              # 빛이 안 나는 positioner 홀 ID. 기본: 분광기에 fiber 가 안 연결된 33 홀, 끊어진 G4 포함 (= cfg.FIBER_EXCLUDE)
FIXED_FIBERS  = []              # 켜져 있지만 움직이지 않는 positioner 홀 ID. 기본: ['G7', 'H8', 'J6'] (sky fiber 가 꽂힌 고정 positioner)
SKY_FIBERS    = []              # 표에서 -9 인 홀의 sky fiber. 기본: ['B7', 'H1', 'N7']
ANGLE_SIGN    = (-1, -1)        # 명령각 = 부호 x 기하각 - 영점. 본 장비 positioner 는 기하각(반시계 alpha, elbow-down beta)의 반대로 돈다 (2026-10-27 확인)
ANGLE_ZERO    = (-30, -150)     # positioner 명령각 영점: 기하각 (30, 150) = 명령 (0, 0) (부호 -1 이라 영점은 -30, -150). 출력·각도 입력 모두 명령각
MEASURED_POSITIONERS = True     # positioner 회전 중심을 arm scan 실측값(Measured_Geometry.txt)으로 쓸지. 핀홀 실측과는 별개 스위치 (2026-10-27 부터 기본 True)
ALPHA_RANGE   = (-330, 30)      # positioner 가 받는 alpha 명령각 범위 [lo, hi). 절대 명령각(Trial 0, 누적각)은 이 범위로 접는다

if FIBER_TABLE:
    os.environ['KSPEC_FIBER_TABLE'] = FIBER_TABLE
os.environ['KSPEC_FIBER_EXCLUDE'] = ','.join(BROKEN_FIBERS)
os.environ['KSPEC_FIBER_FIXED']   = ','.join(FIXED_FIBERS)
os.environ['KSPEC_SKY_FIBERS']    = ','.join(SKY_FIBERS)
os.environ['KSPEC_ANGLE_SIGN']    = f'{ANGLE_SIGN[0]},{ANGLE_SIGN[1]}'
os.environ['KSPEC_ANGLE_ZERO']    = f'{ANGLE_ZERO[0]},{ANGLE_ZERO[1]}'
os.environ['KSPEC_USE_MEASURED']  = '1'                        # fiducial 핀홀 실측 위치 + arm 길이 측정표 (거의 항상 켬)
os.environ['KSPEC_USE_MEASURED_POSITIONERS'] = '1' if MEASURED_POSITIONERS else '0'
os.environ['KSPEC_ALPHA_RANGE']   = f'{ALPHA_RANGE[0]},{ALPHA_RANGE[1]}'
os.makedirs(OUT_DIR, exist_ok=True)
sys.path.insert(0, PKG_DIR)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 80
from kspec_metrology import lab
from kspec_metrology.analysis import fiber_config as cfg
print(f'fiber table {cfg.FIBER_TABLE_PATH}: {len(cfg.fiber_ids())} positioners')
print('broken', cfg.FIBER_EXCLUDE, '| fixed', cfg.FIBER_FIXED, '| sky', cfg.SKY_FIBERS, '| angle sign', cfg.ANGLE_SIGN, '| angle zero', cfg.ANGLE_ZERO)


In [ ]:

MAGNIFICATION  = None           # camera -> focal 배율 초기값. None = 패키지 기본 (2026-01 commissioning fit -8.86, 설치된 실카메라). 10/17 시험 셋업 영상은 8.40
MODEL          = 'full+affine'  # 본 장비. fiducial 30개 기준 leave-one-out 이 가장 좋았다 (README 표). 실험실 광학계는 'radial+affine'
NWINDOW        = 40
NITER_RECENTER = 1
THRESHOLD      = 1e3            # 시작 문턱 [ADU, 배경 포함]; 검출 개수에 맞춰 자동으로 올리고 내린다
MIN_DIST       = 80             # 이보다 가까운 검출은 하나로 합침 [px]
FINDER_OPTS    = {'minarea': 9, 'deblend_cont': 1.0}
N_EXTRA        = 0              # FIXED/SKY 외의 여분 밝은 점 (반사광 등)
SAT_LEVEL      = 65000.
TILE           = 'tile01'



## 1. 목표 파일
`object.info` (xp/yp 또는 각도). 없으면 모든 arm 을 같은 각도로 둔 파일을 만든다 (calibration 배치).


In [ ]:

TARGET_FILE = os.path.join(OUT_DIR, 'exposure_check_target.info')
lab.make_target(TARGET_FILE, angles=(0., 0.))          # 실제 배치의 명령각으로 ((0, 0) = 초기 위치 = 기하각 (30, 150)); 이미 있는 object.info 가 있으면 그 경로를
history = {}



## 2. 검출만 먼저 (선택)


In [ ]:

IMAGE = '/path/to/fib_400ms_0.fits'                    # fiber + fiducial 을 같이 켠 노출 (...0.fits)
Tp = lab.check_peaks(IMAGE, threshold=THRESHOLD, min_dist=MIN_DIST, finder_opts=FINDER_OPTS)



## 3. 종류별 밝기 (노출을 바꿀 때마다 다시 실행)
그림: 영상 위 주황 = fiducial, 하늘색 = fiber (`ID:peak`), 자홍 = sky/고정, 회색 = 기타, 빨간 고리 = 포화. 아래는 종류별 peak 막대 (밝기 순).


In [ ]:

IMAGE = IMAGE                                          # 새로 찍은 이미지면 경로를 바꾼다
T, S, missing = lab.spot_brightness(IMAGE, TARGET_FILE, threshold=THRESHOLD, min_dist=MIN_DIST, finder_opts=FINDER_OPTS,
                                    nwindow=NWINDOW, magnification=MAGNIFICATION, sat_level=SAT_LEVEL, groups=FID_GROUPS if 'FID_GROUPS' in dir() else None)
history[os.path.basename(IMAGE)] = T
display(S.round(0))
T[T.kind != 'fiber'].round(2)


In [ ]:

T[T.kind == 'fiber'].sort_values('peak').round(2)       # fiber 만, 어두운 것부터



### fiducial 그룹별


In [ ]:

lab.plot_group_brightness(T[T.kind == 'fiducial'], cfg.FIDUCIAL_GROUPS, sat_level=SAT_LEVEL, threshold=THRESHOLD, title=os.path.basename(IMAGE));



## 4. 노출 간 비교
같은 종류의 peak 을 노출마다 나란히. 포화되지 않았으면 노출시간에 비례해야 한다.


In [ ]:

cmp = pd.DataFrame({k: T_[T_.kind.isin(['fiber', 'fiducial'])].set_index('id').peak for k, T_ in history.items()})
cmp.insert(0, 'kind', [('fiducial' if i in lab.fiducial_positions()[0] else 'fiber') for i in cmp.index])
cmp.sort_values(['kind', cmp.columns[-1]]).round(0)
